# Scientific Computing Project #1

---------------

**Author:** *Salvador Palma* (mtr765) <br>
**Institution:** UCPH - University of Copenhagen

In [3]:
from watermatrices import Amat, Bmat, yvec
import numpy as np

In [2]:
print(f"Matrix A: {Amat.shape}")
print(f"Matrix B: {Bmat.shape}")
print(f"Vector Y: {yvec.shape}")

Matrix A: (7, 7)
Matrix B: (7, 7)
Vector Y: (7,)


In [12]:
dim = Amat.shape[0]

E = np.block([[Amat, Bmat], [Bmat, Amat]])

I = np.eye(dim)
O = np.zeros((dim, dim))

S = np.block([[I, O], [O, -I]])

z = np.concatenate((yvec, -yvec))

print(f"Matrix E: {E.shape}")
#print(f"Matrix E:\n{E}")
print("\n")

print(f"Matrix S: {S.shape}")
#print(f"Matrix S:\n{S}")
print("\n")

print(f"Vector z: {z.shape}")
print(f"Vector z:\n{z}")

Matrix E: (14, 14)


Matrix S: (14, 14)


Vector z: (14,)
Vector z:
[-0.05677315 -0.00902581  0.16002152  0.07001784  0.67801388 -0.10904168
  0.9050518   0.05677315  0.00902581 -0.16002152 -0.07001784 -0.67801388
  0.10904168 -0.9050518 ]


## Week 1

### a)

#### (1) 

`conditionNumber(M)` follows the equation:

$$ ||A||_\infty = max_i \sum\limits_{j=1}^n |a_{ij}|$$

to compute the matrix norm, and then applies the one given in the exercise:

$$ cond_\infty(M) = ||M||_\infty \cdot ||M^{-1}||_\infty $$

In [47]:
def infNorm(M):
    return np.max(np.sum(np.abs(M), axis=1))
def conditionNumber(M):
    M_inv = np.linalg.inv(M)

    maxValOg = infNorm(M)
    maxValInv = infNorm(M_inv)
    
    return maxValOg * maxValInv

#### (2)

Given that the only non-exact input is the right-hand side `z`, with 8 significant digits, its relative error is bounded by $\frac{||\Delta z||_\infty}{||z||_\infty} \le \frac{1}{2}10^{-8}$, and this error associates to the solution as

$$\frac{||\Delta x||_\infty}{||x||_\infty} \le \text{cond}_\infty(E-\omega S)\cdot\frac{||\Delta z||_\infty}{||z||_\infty} \le \text{cond}_\infty(E-\omega S)\cdot \frac{1}{2}10^{-8}$$

This means that the condition number acts as an amplification factor, that is, it measures how much a relative error in the input increases in the output. To calculate the amount of decimal digits we can guarantee in the solution `x`, we can leverage the following formula based on the relative error just calculated:

$$ \text{Guaranteed Digits} = \lfloor-\log_{10} |E_{rel}|\rfloor$$

which leads us to the results depicted below:

| $\omega$ | $\text{cond}_\infty(E- \omega S) \approx$ | upper bound | guaranteed digits in `x` |
| --- | --- | --- | --- |
| 0.800 | 327.82 | 0.00000 | **5** |
| 1.146 | 152679.27 | 0.00076 | **3** |
| 1.400 | 227.19 | 0.00000 | **5** |

In [62]:
ws = [0.800, 1.146, 1.400]

for w in ws:
    M = E - w * S
    condN = conditionNumber(M)
    print(f"w={w:.3f}")
    print(f"condition number = {condN:.5f}")
    print(f"upper bound = {condN*0.5e-8:.3e}")
    print(f"guaranteed digits = {np.floor(-np.log10(abs(condN*0.5e-8)))}")
    print("\n")

w=0.800
condition number = 327.81670
upper bound = 1.639e-06
guaranteed digits = 5.0


w=1.146
condition number = 152679.26875
upper bound = 7.634e-04
guaranteed digits = 3.0


w=1.400
condition number = 227.19444
upper bound = 1.136e-06
guaranteed digits = 5.0




### b)

#### (1)

For each $\omega$, we determine the bound on the relative forward error via:

$$ \frac{||\Delta x||_\infty}{||\hat{x}||_\infty} \le \text{cond}_\infty (E - \omega S) \cdot \frac{||\delta \omega S||_\infty}{||E - \omega S||_\infty}$$

with $\delta \omega$ being $\frac{1}{2} \cdot 10^{-3}$, the bounds are as follows:

| $\omega$ | bound |
|---|---|
| 0.800 | 0.00522|
| 1.146 | 2.40504 |
| 1.400 | 0.00355 |

In [52]:
for w in ws:
    M = E - w * S
    condN = conditionNumber(M)
    upper = infNorm(0.0005 * S)
    lower = infNorm(M)
    upperBound = condN * (upper / lower)
    print(f"w={w:.3f} upper bound = {upperBound:.5f}")


w=0.800 upper bound = 0.00522
w=1.146 upper bound = 2.40504
w=1.400 upper bound = 0.00355


#### (2)

Using the upper bounds for the relative errors that we just calculated, and the following formula:

$$ \text{Guaranteed Digits} = \lfloor-\log_{10} |E_{rel}|\rfloor$$

We can calculate the amount of significant digits that we can guarantee in `x`, leading to the following results

| $\omega$ | guaranteed digits |
|---|---|
| 0.800 | 2|
| 1.146 | 0 |
| 1.400 | 2 |

*Note: $\omega=1.146$ actually leads to the equation above to equal $-1$, but this is the same as saying we can't guarantee **any** decimal digit*

In [51]:
for w in ws:
    M = E - w * S
    condN = conditionNumber(M)
    upper = infNorm(0.0005 * S)
    lower = infNorm(M)
    upperBound = condN * (upper / lower)
    
    guaranteedDigits = np.floor(-np.log10(abs(upperBound)))
    guaranteedDigits = max(0, guaranteedDigits)
    print(f"w={w:.3f} guaranteed digits = {guaranteedDigits:.0f}")

w=0.800 guaranteed digits = 2
w=1.146 guaranteed digits = 0
w=1.400 guaranteed digits = 2
